# Base-learner internals

Each `PreparedLearner` variant (`learners.rs:463`) trades generality for speed
in the boosting loop. The engine calls `fit_step_ref`, `predict_into` and
`reweight` on each variant. Variants cache different quantities at preparation
time and use different algorithms for each update.
This chapter goes variant by variant, then collects the cost model and the
cache-vs-recompute table.

Read [](architecture.ipynb) first for the enum's role in the engine loop and the
methods each variant supplies.

## `ScalarBols` — single-column linear term

```text
ScalarBols {
    design: Array1<f64>,            // centered column x̃ (length n)
    weighted_design_t: Array1<f64>, // W x̃
    scalar_denom: f64,              // x̃ᵀ W x̃ + λ  (scalar)
    penalty: f64,                   // λ (already solved from df)
    centering: f64,                 // x̄ (to uncenter at coef time)
    beta_buf: Vec<f64>,             // length-1 scratch
    pred_buf: Vec<f64>,             // length-n scratch
    ...
}
```

The common case in `glmboost`. The "design" is one centered column, so the
penalised solve degenerates to a scalar:

$$
\hat\beta = \frac{\tilde x^\top W u}{\tilde x^\top W \tilde x + \lambda}.
$$

The learner caches $W \tilde x$ and the scalar denominator, so a `fit_step_ref`
is **one dot product and one division** — no matrix at all. This is why
`glmboost` over thousands of columns is fast: per learner per iteration is
$O(n)$, not $O(n p)$.

Centering means are stored (`centering = x̄`) so `coef()` can shift the
coefficient back to the uncentered scale: $\beta_0^{\text{unc}} = \hat f^{[0]} -
\beta_1^{\text{agg}} \bar x$.

## `BlockBols` — multi-column linear / spline term

```text
BlockBols {
    design: Array2<f64>,            // X (n × p_j)
    wdt_faer: Mat<f64>,             // W Xᵀ in column-major faer form (p_j × n)
    design_faer: Mat<f64>,          // X in column-major faer form
    solver: GramSolver,             // factorised (Xᵀ W X + λ K) — see linear_algebra.md
    penalty: Array2<f64>,           // K (baked with λ at prepare time)
    centering: Array1<f64>,         // column means of X
    rhs_buf, beta_buf, pred_buf,    // per-learner scratch
    ...
}
```

Used for factor `bols`, `bbs`, `bns`, `bspatial`, `bmrf`, `brandom` — anything
with a multi-column design and a penalty matrix. The per-step solve is a
`GramSolver` back-substitution (`solve_into_slice`):

$$
G \hat\beta = X^\top W u, \qquad G = X^\top W X + \lambda K, \qquad O(p_j^2).
$$

The factorisation of $G$ (Cholesky, with jitter fallback) is done **once** at
`prepare_*` time and reused for every boosting step and every CV fold's
back-solve.

`bbs` is the most common `BlockBols` user. Construction
(`prepare_bbs_with_lambda`) builds the B-spline basis, the difference penalty
$K_1 = \Delta_d^\top \Delta_d$ of order $d$ (default 2), solves `df` $\rightarrow$ `λ`,
bakes `λ` into the penalty, then hands off to `prepare_block_bols`. Boundary
knots and the difference-penalty order are stored so the basis can be rebuilt
on new data at predict time.

## `BmonoBbs` — monotone / shape-constrained spline

Carries the spline design and difference penalty (as `BlockBols` does) plus a
constrained solver. The per-step problem is

$$
\min_\beta \tfrac12 \beta^\top A \beta - b^\top \beta \quad \text{s.t.} \quad D\beta \ge 0,
\qquad A = X^\top W X + \lambda K, \quad b = X^\top W u.
$$

The constraint matrix $D$ encodes the shape restriction. The supported
constraints (`MonoConstraint`, `learners.rs:365`):

| Variant | Constraint | Spline shape |
|---|---|---|
| `Increasing` | $\beta_{i+1} \ge \beta_i$ | non-decreasing |
| `Decreasing` | $\beta_{i+1} \le \beta_i$ | non-increasing |
| `Convex` | $\Delta^2 \beta \ge 0$ | convex |
| `Concave` | $\Delta^2 \beta \le 0$ | concave |
| `Positive` | $\beta \ge 0$ | non-negative |
| `Negative` | $\beta \le 0$ | non-positive |
| `Constrained` | user-supplied $D$ | arbitrary |

### Two solver modes (`BmonoSolver`, `learners.rs:384`)

- **`Iterative`** (R `type = "iterative"`, not the default): a
  violation-penalty fixed point. At each inner iteration, augment $A$ with $\lambda_2 \cdot D^\top
  \operatorname{diag}(V) D$ where $V_i = 1$ for each currently-violated
  constraint, re-solve the unconstrained problem with the existing
  `GramSolver`, refresh $V$. Converges when $V$ stops changing or `niter`
  (default 50) is reached. $\lambda_2$ defaults to $10^6$ — large enough that
  violations are penalised to $\approx$ machine precision on well-conditioned
  problems but small enough to keep $A$ numerically well-conditioned.
- **`QuadProg`** (R `type = "quad.prog"`): a primal active-set QP. Slower per
  step but produces a strictly KKT-feasible $\beta$. Use when the constraint
  must be enforced exactly (mis-specified models, sparse data, downstream uses
  that depend on feasibility).

The trade-off: the iterative solver is good enough for the boosting inner
loop, where the constrained solve is one step in a long path and small
projection error is dominated by $m_{\text{stop}}$ noise. The QP exists for
users who need exact feasibility.

### `Bconstrained`

`Bconstrained` (`AGENTS.md` gotchas 30–32) is `bmono` with
`MonoConstraint::Constrained` and a user-supplied $C$ matrix as $D$. It
collects data into the same lists as `bmono` in the Python front-end
(`gamboost.py:2199`) and reuses the entire `BmonoBbs` prediction pipeline, so
predict-with-newdata, cvrisk, and `partial_effects` all work for
`Bconstrained` (unlike `bkernel` / `buser`, which raise `NotImplementedError`
in some paths).

The constraint matrix $C$ must match the B-spline column count
(`learners.rs:2171`): `2*(degree+1) + n_inner` knots give
`n_inner + degree + 1` columns. For default `knots=20`, `degree=3`, that is
24 columns, so `C = np.eye(24)` is the smallest valid constraint.

## `Btree` — tree stump

```text
Btree {
    design: Array2<f64>,            // X (n × p)
    minbucket, minsplit, maxdepth,  // R control parameters
    by: Option<Array1<f64>>,        // optional by-variable
    beta_buf: Vec<f64>,             // length-4 (the encoded stump)
    pred_buf: Vec<f64>,             // length-n
    ...
}
```

Each step searches splits across the input columns for the best weighted-SSE
stump (or a shallow tree up to `maxdepth`), returns node means as the
prediction, and encodes the fitted tree compactly in the coefficient vector:

```{figure} ../figures/mermaid-232bbeb3595e.svg
:name: mermaid-232bbeb3595e
:alt: Tree base-learner pipeline from pseudo-residuals through split search and compact coefficient encoding to prediction.

A tree base learner scans candidate splits, stores the winning split and node means, and replays that compact representation during prediction.
```

The 4-number encoding `[split_var_idx, split_point, left_mean, right_mean]` is
what `predict_into` replays on new data. `Btree` exposes no per-learner
hat matrix — a tree's smoother depends on the partition fitted at the
step — but conditional on that partition the step is a weighted
averaging of the working response over each leaf, and the hat-matrix
recursion builds the step's matrix from the recorded coefficients
(`btree_step_hat_matrix`). df and AIC are therefore available for fits
containing tree terms, under the conditional convention the smooth
learners already get: the matrix is conditional on the fitted
partition, and split selection is not counted. It also disables the
Gaussian fast path (it is not a
linear smoother), forcing the general loop.

## `KroneckerBols` — multinomial / array models

```text
KroneckerBols {
    x1: Array2<f64>,                // covariate design X₁ (n × p₁)
    x1_faer, wx1t_faer: Mat<f64>,   // faer forms
    solver: GramSolver,             // factorised (X₁ᵀ W X₁ + λ K₁)
    x2: Array2<f64>,                // class design X₂ (typically I_{K-1})
    ...
}
```

Implements the Kronecker-product fit for matrix responses (multinomial). The
combined design is $X_2 \otimes X_1$ over the stacked response; with $X_2 = I$
the penalised normal equations are **block-diagonal**, so the fit is $K-1$
independent ridge solves sharing a single factorisation of $X_1^\top W X_1 +
P$. A step solves $K-1$ systems sharing that one `GramSolver`, then assembles
the prediction $\operatorname{vec}(X_1 \beta X_2^\top)$.

The penalty detail ($P = I + K_1$, the `+I` term from R's `%O%`) and its
derivation are documented in [](kronecker_multinomial.ipynb).

## Per-variant cost model

The per-step cost (one iteration, one learner) of each variant:

| Variant | Prepare cost | Per-step `fit_step_ref` | `predict_into` |
|---|---|---|---|
| `ScalarBols` | $O(n)$ | $O(n)$ — dot product + division | $O(n)$ |
| `BlockBols` | $O(n p_j^2 + p_j^3)$ — build Gram + factorise | $O(n p_j + p_j^2)$ — matvec + back-solve | $O(n p_j)$ |
| `BmonoBbs` (iterative) | as `BlockBols` | $O(\text{niter} \cdot p_j^2)$ inner re-solves | $O(n p_j)$ |
| `BmonoBbs` (QP) | as `BlockBols` | $O(p_j^3)$ active-set worst case | $O(n p_j)$ |
| `Btree` | $O(n p)$ — sort columns once | $O(n p)$ — split scan | $O(n)$ |
| `KroneckerBols` | $O(n p_1^2 + p_1^3)$ | $O((K{-}1)(n p_1 + p_1^2))$ | $O((K{-}1) n p_1)$ |

A full iteration adds $J$ per-learner solves plus one $O(n)$ `axpy` to update
the predictor. The Gaussian fast path ([](engine.ipynb)) removes the per-learner
$O(n p_j)$ projection for linear smoothers, replacing it with $O(p_w p_{j^\star})$
cross-Gram updates — a factor-of-$J$ speedup when $J$ is large.

## Cached and recomputed quantities

| Quantity | When computed |
|---|---|
| design $X$, penalty $K$, basis knots | once, at `prepare_*` |
| `λ` from `df` | once at prepare; again per CV fold (weights change) |
| penalised Gram factorisation | once at prepare; rebuilt on `reweight` |
| $W X^\top$ | once at prepare |
| constraint matrix $D$ (`bmono`) | once at prepare |
| coefficient, prediction, SSE | every boosting step (cheap back-solve + matvec) |
| split scan (`btree`) | every boosting step |

Preparation computes quantities that do not depend on the pseudo-residual `u`.
Each step then runs the `u`-dependent back-solve without allocating on the heap,
including in a 1,000-iteration fit.

## See also

- [](architecture.ipynb) — the engine methods each variant supplies.
- [](linear_algebra.ipynb) — the `GramSolver` used by every variant except
  `ScalarBols` and `Btree`.
- [](kronecker_multinomial.ipynb) — the full Kronecker reconstruction.
- [](../methodology/base_learners) — the statistical motivation and model
  interpretation of each learner type.
- [](../methodology/penalties_df) — how effective degrees of freedom define
  the penalty strength.
- [](../usage/base_learners) — runnable code examples for every learner.